In [1]:
import sys
import os
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, str(root_dir))   


############################################## Dataset
# from datasets.rescuenet import Dataset
from datasets.floodnet import Dataset
# from datasets.cracks import Dataset
######################################################

import torch
from tqdm import tqdm 

In [2]:
# #---------------------------------------- rescuenet
# Dataset.stats_from_yaml('rescuenet.yaml')
# dataset = Dataset(split='test', scale=3000)

#---------------------------------------- floodnet
Dataset.stats_from_yaml('floodnet.yaml')
dataset = Dataset(split='test', scale=704)

# #---------------------------------------- public cracks
# subfolders = [
#               'ConcreteCrack', 
#               #'SyntheticCracks',
#               'Stone331',
#               'CrackTree260', 
#               'DeepCrack', 
#               'CrackLS315',         
#               'CrackForest',             
#               'CRKWH100', 
#               ]  

# public = "public-cracks"
# Dataset.stats_from_yaml('cracks-public.yaml')
# dataset = Dataset(split='test', root_path=public, subfolders=subfolders)


# # --------------------------------------- photos
# photos ="Dataset_2D_Train_Undersampled_No_CastelNuovo_AreaGrande_CentroItalia"
# Dataset.stats_from_yaml('cracks-photos.yaml')
# dataset = Dataset(split='test', root_path=photos)


# #---------------------------------------- textures
# textures =  "Dataset_Textures_Train_Undersampled_2.0"
# Dataset.stats_from_yaml('cracks-textures.yaml')
# dataset = Dataset(split='test', root_path=textures)


# #---------------------------------------------- textures + photos : Crack-concat
# Dataset.stats_from_yaml('cracks-concat.yaml')
# concat =  "Cracks-concat"
# dataset = Dataset(split='train', root_path=concat)


In [3]:

loader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=False)
#if batch_size != 1, the program will crash because the images are not 
# of the same size and cannot stay in the same batch

num_labels = len(dataset.class_names)
print('dataset length:', len(dataset))
print('number of labels:', num_labels)
print('class names:', dataset.class_names)

dataset length: 448
number of labels: 10
class names: ['Background', 'Building-flooded', 'Building-not-flooded', 'Road-flooded', 'Road-not-flooded', 'Water', 'Tree', 'Vehicle', 'Pool', 'Grass']


In [4]:
def count_labels_and_pixels(data_loader, num_labels):
    label_presence = torch.zeros(num_labels, dtype=torch.long)
    pixel_counts = torch.zeros(num_labels, dtype=torch.long)

    counter = 0
    # Iterate over batches of images
    for data, target in tqdm(data_loader, desc="Counting"):
        #print(target.shape, target.numel())
        #n = target.numel()
        for label in target:
            #print("Current value:", counter := counter + 1) 
            #print(label.shape, label.numel())
            #print(label, type(label), label.dtype)
            if label.dtype != torch.int64:
                label = label.long()
            unique_labels, counts = torch.unique(label, return_counts=True)
            #print(unique_labels, counts)
            label_presence[unique_labels] += 1
            for label, count in zip(unique_labels, counts):
                pixel_counts[label] += count 
        #print(label_presence, pixel_counts)

    return label_presence, pixel_counts

image_counts, pixel_counts = count_labels_and_pixels(loader, num_labels)

print("Number of images each label appears in:", image_counts)
print("Number of pixels for each label:", pixel_counts)


Counting: 100%|██████████| 448/448 [00:14<00:00, 31.16it/s]

Number of images each label appears in: tensor([ 40,  47, 173,  49, 234, 183, 364, 166,  96, 414])
Number of pixels for each label: tensor([  6440544,   5418099,  10253221,   7318655,  17874346,  32682096,
         53758369,    493542,    572781, 166242683])
